In [ ]:
import os
import math
import joblib
import pandas as pd

In [ ]:
model_path = "../models/crop_recommendation_model.pkl"
dataset_path = "../data/Crop_recommendation.csv"

if not os.path.exists(model_path):
    raise FileNotFoundError(
        "Model file not found. Please check the models folder."
    )

saved_data = joblib.load(model_path)

if isinstance(saved_data, dict):

    if "model" not in saved_data or "features" not in saved_data:
        raise ValueError(
            "The saved model package is missing required information."
        )

    model = saved_data["model"]
    features = saved_data["features"]

    input_ranges = saved_data.get("input_ranges")

else:
    model = saved_data
    features = None
    input_ranges = None


# Load the dataset only when needed
if features is None or input_ranges is None:

    if not os.path.exists(dataset_path):
        raise FileNotFoundError(
            "Dataset not found. Please check the project folder."
        )

    df = pd.read_csv(dataset_path)

    if features is None:
        features = [
            column for column in df.columns
            if column != "label"
        ]

    if input_ranges is None:
        input_ranges = {
            feature: (
                float(df[feature].min()),
                float(df[feature].max())
            )
            for feature in features
        }


print("Model loaded successfully!")
print("Model type:", type(model).__name__)
print("Features:", features)

Model loaded successfully!
Model type: RandomForestClassifier
Features: ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']


In [ ]:
VALID_RANGES = input_ranges

print("\nAccepted Input Ranges")
print("=" * 45)

for feature in features:
    minimum, maximum = VALID_RANGES[feature]

    print(
        f"{feature}: {minimum:.2f} to {maximum:.2f}"
    )


Accepted Input Ranges
N: 0.00 to 140.00
P: 5.00 to 145.00
K: 5.00 to 205.00
temperature: 8.83 to 43.68
humidity: 14.26 to 99.98
ph: 3.50 to 9.94
rainfall: 20.21 to 298.56


In [ ]:
def get_valid_input(feature, minimum, maximum):

    while True:

        try:
            user_value = input(
                f"Enter {feature} "
                f"({minimum:.2f} to {maximum:.2f}): "
            ).strip()

            if not user_value:
                print(
                    "Invalid input! Input cannot be empty. "
                    "Please try again.\n"
                )
                continue

            value = float(user_value)

            if not math.isfinite(value):
                print(
                    "Invalid input! Please enter a finite number.\n"
                )
                continue

            if not minimum <= value <= maximum:
                print(
                    f"Invalid input! {feature} must be between "
                    f"{minimum:.2f} and {maximum:.2f}. "
                    "Please enter the value again.\n"
                )
                continue

            return value

        except ValueError:
            print(
                "Invalid input! Please enter a numeric value.\n"
            )

In [ ]:
def get_user_input():

    print("\n" + "=" * 45)
    print("       CROP RECOMMENDATION SYSTEM")
    print("=" * 45)

    print("Enter the soil and environmental details.\n")

    # Required input sequence
    input_sequence = [
        "N",
        "P",
        "K",
        "temperature",
        "humidity",
        "ph",
        "rainfall"
    ]

    user_data = {}

    for feature in input_sequence:

        # Skip features not required by the trained model
        if feature not in features:
            continue

        minimum, maximum = VALID_RANGES[feature]

        user_data[feature] = get_valid_input(
            feature,
            minimum,
            maximum
        )

    input_df = pd.DataFrame([user_data])

    # Match the exact feature order expected by the model
    input_df = input_df[features]

    return input_df

In [ ]:
def predict_crop(input_df):

    predicted_crop = model.predict(input_df)[0]

    print("\n" + "=" * 45)
    print("          CROP PREDICTION RESULT")
    print("=" * 45)

    print("Recommended Crop:", str(predicted_crop).upper())

    print("=" * 45)

    return predicted_crop

In [ ]:
user_input = get_user_input()

predicted_crop = predict_crop(user_input)


       CROP RECOMMENDATION SYSTEM
Enter the soil and environmental details.

Invalid input! Please enter a numeric value.

Invalid input! Input cannot be empty. Please try again.

Invalid input! Input cannot be empty. Please try again.

Invalid input! Input cannot be empty. Please try again.

Invalid input! Input cannot be empty. Please try again.

Invalid input! Input cannot be empty. Please try again.

Invalid input! K must be between 5.00 and 205.00. Please enter the value again.

